# Monitor — YF model *in silico* X-scan

Builds `Xscan.csv` from `TableS10_X-scan_oriDATA.xlsx`.

Output schema matches `260617_test_ESMFold2/benchmark/positives/AF3_class_I.csv`:

`id, cdr3_TRA, cdr3_TRB, TRAV, TRAJ, TRBV, TRBJ, species, peptide, MHC_allele_a`

plus two extra columns: **`ActivationScore`** (experimental read-out) and `tcr` (source sheet,
kept for grouping — drop it if the downstream builder is strict about columns).

Each xlsx sheet is one TCR (19 aa x 9 positions = 171 rows). The 9 rows per sheet where the
substituted residue equals the wild-type residue carry no `ActivationScore` and reproduce the
WT peptide; they are collapsed into a single WT row per TCR (see `INCLUDE_WT`).

In [1]:
import pandas as pd
from pathlib import Path

WD   = Path("/Users/roessner/Documents/PostDoc/Data/261001_YF_model_in_silico_Xscan")
XLSX = WD / "TableS10_X-scan_oriDATA.xlsx"
OUT  = WD / "Xscan.csv"

MHC_ALLELE_A = "HLA_A0201"
SPECIES      = "HomoSapiens"

# Keep one wild-type peptide row per TCR (ActivationScore = NA).
INCLUDE_WT = True

# Per-sheet TCR chain definitions.
TCRS = {
    "TCR_YF":  dict(TRAV="TRAV12-2", cdr3_TRA="CAVGDDKIIF",  TRAJ="TRAJ30",
                    TRBV="TRBV28",   cdr3_TRB="CASTPQTAYEQYF", TRBJ="TRBJ2-7"),
    "TCR_YF2": dict(TRAV="TRAV12-2", cdr3_TRA="CAVNPDKIIF",  TRAJ="TRAJ30",
                    TRBV="TRBV4-2",  cdr3_TRB="CASSQEDRGPEKLFF", TRBJ="TRBJ1-4"),
    "TCR_YF3": dict(TRAV="TRAV12-2", cdr3_TRA="CAAGDDKIIF",  TRAJ="TRAJ30",
                    TRBV="TRBV29-1", cdr3_TRB="CSVATSGGSNEQFF", TRBJ="TRBJ2-1"),
}

COLS = ["id","cdr3_TRA","cdr3_TRB","TRAV","TRAJ","TRBV","TRBJ",
        "species","peptide","MHC_allele_a","ActivationScore","tcr"]

## 1. Read sheets and check assumptions

In [2]:
xl = pd.ExcelFile(XLSX)
print("sheets in file:", xl.sheet_names)

missing = set(TCRS) - set(xl.sheet_names)
assert not missing, f"sheets defined but absent from xlsx: {missing}"
extra = set(xl.sheet_names) - set(TCRS)
if extra:
    print(f"WARNING: sheets present but not defined, skipped: {sorted(extra)}")

raw = {}
for sheet in TCRS:
    df = xl.parse(sheet)
    raw[sheet] = df
    wt = df.loc[df.ActivationScore.isna(), "epitope"].unique()
    print(f"{sheet:8s} rows={len(df):4d}  aa={df.Aa.nunique()}  pos={df.Position.nunique()}  "
          f"unique_peptides={df.epitope.nunique():4d}  NA_score={df.ActivationScore.isna().sum()}  WT={wt}")
    assert set(df.epitope.str.len()) == {9}, f"{sheet}: non-9mer peptide present"

sheets in file: ['TCR_YF', 'TCR_YF2', 'TCR_YF3']
TCR_YF   rows= 171  aa=19  pos=9  unique_peptides= 163  NA_score=9  WT=<StringArray>
['LLWNGPMAV']
Length: 1, dtype: str
TCR_YF2  rows= 171  aa=19  pos=9  unique_peptides= 163  NA_score=9  WT=<StringArray>
['LLWNGPMAV']
Length: 1, dtype: str
TCR_YF3  rows= 171  aa=19  pos=9  unique_peptides= 163  NA_score=9  WT=<StringArray>
['LLWNGPMAV']
Length: 1, dtype: str


## 2. Assemble one table per TCR

In [3]:
records = []

for sheet, chains in TCRS.items():
    df = raw[sheet].copy()

    # Rows with no ActivationScore are the WT-for-WT substitutions -> the WT peptide.
    is_wt = df.ActivationScore.isna()
    wt_peptides = df.loc[is_wt, "epitope"].unique()
    assert len(wt_peptides) <= 1, f"{sheet}: ambiguous WT peptide {wt_peptides}"

    mut = df.loc[~is_wt, ["epitope", "ActivationScore"]].copy()
    assert not mut.epitope.duplicated().any(), f"{sheet}: duplicated mutant peptide"

    if INCLUDE_WT and len(wt_peptides) == 1:
        mut = pd.concat(
            [pd.DataFrame({"epitope": wt_peptides, "ActivationScore": [pd.NA]}), mut],
            ignore_index=True,
        )

    block = pd.DataFrame({
        "cdr3_TRA":        chains["cdr3_TRA"],
        "cdr3_TRB":        chains["cdr3_TRB"],
        "TRAV":            chains["TRAV"],
        "TRAJ":            chains["TRAJ"],
        "TRBV":            chains["TRBV"],
        "TRBJ":            chains["TRBJ"],
        "species":         SPECIES,
        "peptide":         mut.epitope.values,
        "MHC_allele_a":    MHC_ALLELE_A,
        "ActivationScore": mut.ActivationScore.values,
        "tcr":             sheet,
    })
    records.append(block)
    print(f"{sheet:8s} -> {len(block):4d} rows (WT included: {INCLUDE_WT and len(wt_peptides)==1})")

xscan = pd.concat(records, ignore_index=True)
xscan.insert(0, "id", [f"tcr_{i:04d}" for i in range(1, len(xscan) + 1)])
xscan = xscan[COLS]
print("\ntotal rows:", len(xscan))
xscan.head()

TCR_YF   ->  163 rows (WT included: True)
TCR_YF2  ->  163 rows (WT included: True)
TCR_YF3  ->  163 rows (WT included: True)

total rows: 489


,id,cdr3_TRA,cdr3_TRB,TRAV,TRAJ,TRBV,TRBJ,species,peptide,MHC_allele_a,ActivationScore,tcr
0,tcr_0001,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLWNGPMAV,HLA_A0201,<NA>,TCR_YF
1,tcr_0002,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,KLWNGPMAV,HLA_A0201,99.73,TCR_YF
2,tcr_0003,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LKWNGPMAV,HLA_A0201,0.08,TCR_YF
3,tcr_0004,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLKNGPMAV,HLA_A0201,-0.06,TCR_YF
4,tcr_0005,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLWKGPMAV,HLA_A0201,12.38,TCR_YF


## 3. Validate against the reference schema

In [4]:
REF = Path("/Users/roessner/Documents/PostDoc/Data/260617_test_ESMFold2/"
            "benchmark/positives/AF3_class_I.csv")

ref_cols = list(pd.read_csv(REF, nrows=0).columns)
print("reference columns:", ref_cols)
print("our columns     :", list(xscan.columns))

assert list(xscan.columns[:len(ref_cols)]) == ref_cols, "leading columns deviate from reference"
assert xscan.id.is_unique, "ids not unique"
assert not xscan.peptide.isna().any(), "missing peptide"
assert xscan.groupby("tcr").peptide.apply(lambda s: s.is_unique).all(), "peptide repeated within a TCR"

print("\nrows per TCR:")
print(xscan.tcr.value_counts().sort_index().to_string())
print("\nActivationScore: NA =", int(xscan.ActivationScore.isna().sum()),
      "| min =", xscan.ActivationScore.min(), "| max =", xscan.ActivationScore.max())

reference columns: ['id', 'cdr3_TRA', 'cdr3_TRB', 'TRAV', 'TRAJ', 'TRBV', 'TRBJ', 'species', 'peptide', 'MHC_allele_a']
our columns     : ['id', 'cdr3_TRA', 'cdr3_TRB', 'TRAV', 'TRAJ', 'TRBV', 'TRBJ', 'species', 'peptide', 'MHC_allele_a', 'ActivationScore', 'tcr']

rows per TCR:
tcr
TCR_YF     163
TCR_YF2    163
TCR_YF3    163

ActivationScore: NA = 3 | min = -0.17 | max = 180.69


## 4. Write `Xscan.csv`

In [5]:
xscan.to_csv(OUT, index=False)
print("wrote", OUT, f"({len(xscan)} rows, {len(xscan.columns)} columns)")
print(OUT.read_text().split("\n")[0])
pd.read_csv(OUT).head()

wrote /Users/roessner/Documents/PostDoc/Data/261001_YF_model_in_silico_Xscan/Xscan.csv (489 rows, 12 columns)
id,cdr3_TRA,cdr3_TRB,TRAV,TRAJ,TRBV,TRBJ,species,peptide,MHC_allele_a,ActivationScore,tcr


,id,cdr3_TRA,cdr3_TRB,TRAV,TRAJ,TRBV,TRBJ,species,peptide,MHC_allele_a,ActivationScore,tcr
0,tcr_0001,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLWNGPMAV,HLA_A0201,NaN,TCR_YF
1,tcr_0002,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,KLWNGPMAV,HLA_A0201,99.73,TCR_YF
2,tcr_0003,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LKWNGPMAV,HLA_A0201,0.08,TCR_YF
3,tcr_0004,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLKNGPMAV,HLA_A0201,-0.06,TCR_YF
4,tcr_0005,CAVGDDKIIF,CASTPQTAYEQYF,TRAV12-2,TRAJ30,TRBV28,TRBJ2-7,HomoSapiens,LLWKGPMAV,HLA_A0201,12.38,TCR_YF
